# Week 7 — Anatomy of a Physics Engine
### Complete, runnable reference code

**Physics 311: Mechanics — Modeling, Computing, Publishing**

---

This notebook is the executable companion to the Week 7 lesson. It contains the **entire engine** in working form, plus the validation runs that prove it reproduces Labs 1&ndash;6.

Run it top to bottom (**Kernel &rarr; Restart & Run All**). Every printed number below was produced by executing these exact cells.

> **This is a reference, not the lab.** Lab 7 asks you to build *your own* engine. Read this one for the architecture, then write yours. Code you copy without understanding becomes a liability in Block 3, when you have to port it to JavaScript line by line.

---

## The one idea

Six labs, one repeated skeleton: *set up state &rarr; loop over time &rarr; compute forces &rarr; integrate &rarr; record &rarr; plot.* You wrote that skeleton six times.

An **engine** writes it once and lets you plug in what changes.

| What changed between labs | Component that owns it |
|---|---|
| The objects (mass, radius, position, orientation) | `Body` |
| The interactions (gravity, springs, drag, contact) | `Force` objects in a registry |
| The update rule (Euler, Euler-Cromer, Verlet) | `Integrator` |
| Time stepping and bookkeeping | `World` |

The test of success: **a new simulation should be a new `main`, not a new engine.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ellipk

plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
UT_ORANGE = '#ff8200'

print("NumPy", np.__version__)

---
## 1 &nbsp; `Body` — what the world is made of

A body owns **state** (position, velocity, orientation, spin) and **parameters** (mass, radius, moment of inertia). It owns no physics: it does not know what forces exist or how time advances.

Three design decisions worth copying:

- **`f` and `tau` are accumulators.** They are zeroed at the start of each step, every force *adds* to them, then the integrator consumes them. No force ever needs to know what other forces exist.
- **`inv_m` and `inv_I` are properties, not stored numbers.** A fixed body returns inverse mass `0.0`, which makes it immovable *arithmetically* &mdash; no `if body.fixed` branches scattered through your collision code.
- **The body computes its own `ke()`, `p()`, `L_about()`.** Diagnostics live with the data they describe.

In [ ]:
class Body:
    """A rigid disk: linear state + optional rotational state."""

    def __init__(self, m=1.0, r=(0.0, 0.0), v=(0.0, 0.0), radius=0.0,
                 I=None, theta=0.0, omega=0.0, fixed=False, name=""):
        self.m      = float(m)
        self.r      = np.array(r, dtype=float)     # position  (m)
        self.v      = np.array(v, dtype=float)     # velocity  (m/s)
        self.radius = float(radius)                # for contact tests
        self.I      = I                            # moment of inertia; None = point mass
        self.theta  = float(theta)                 # orientation (rad)
        self.omega  = float(omega)                 # angular velocity (rad/s)
        self.fixed  = bool(fixed)                  # immovable?
        self.name   = name

        self.f   = np.zeros(2)   # force accumulator  -- cleared every step
        self.tau = 0.0           # torque accumulator -- cleared every step

    # --- inverse mass makes "fixed" arithmetic, not conditional ---
    @property
    def inv_m(self):
        return 0.0 if self.fixed else 1.0 / self.m

    @property
    def inv_I(self):
        if self.fixed or self.I is None or self.I == 0:
            return 0.0
        return 1.0 / self.I

    def clear(self):
        self.f[:] = 0.0
        self.tau  = 0.0

    def add_force(self, F):
        self.f += np.asarray(F, dtype=float)

    # --- diagnostics ---
    def ke(self):
        k = 0.5 * self.m * self.v.dot(self.v)
        if self.I:
            k += 0.5 * self.I * self.omega**2
        return k

    def p(self):
        return self.m * self.v

    def L_about(self, origin=(0.0, 0.0)):
        d  = self.r - np.asarray(origin, dtype=float)
        Lz = self.m * (d[0]*self.v[1] - d[1]*self.v[0])     # orbital
        if self.I:
            Lz += self.I * self.omega                        # spin
        return Lz

    def __repr__(self):
        return f"Body({self.name or 'body'}, m={self.m}, r={self.r}, v={self.v})"


b = Body(m=2.0, r=(1, 2), v=(3, -1), radius=0.5, I=0.25, omega=4.0, name="test")
print(b)
print(f"  KE        = {b.ke():.4f} J")
print(f"  p         = {b.p()}")
print(f"  L (origin)= {b.L_about():.4f}")
print(f"  inv_m     = {b.inv_m:.4f}")

wall = Body(m=1.0, fixed=True, name="wall")
print(f"\n  fixed body inv_m = {wall.inv_m}  <- immovable by arithmetic")

---
## 2 &nbsp; `Force` — the registry and the polymorphism trick

Every force exposes the same two-method interface:

```python
force.apply(world)              # add to body.f / body.tau
force.potential_energy(world)   # 0.0 if non-conservative
```

The world then does this, and never asks what kind of force it is holding:

```python
for f in self.forces:
    f.apply(self)
```

That one loop is why adding a new force never requires editing `World`. It is the single most important consequence of the architecture.

**Conservative forces return a potential energy; dissipative ones return `0.0`.** That default is what makes `world.total_energy()` meaningful *and* makes energy drift a genuine diagnostic: if drag is present, energy *should* fall.

In [ ]:
class Force:
    """Interface. Subclass and override apply()."""
    def apply(self, world):
        raise NotImplementedError
    def potential_energy(self, world):
        return 0.0          # non-conservative by default


class Gravity(Force):
    """Uniform gravitational field (Labs 1-2)."""
    def __init__(self, g=9.81):
        self.gmag = g
        self.g    = np.array([0.0, -g])
    def apply(self, world):
        for b in world.bodies:
            if not b.fixed:
                b.add_force(b.m * self.g)
    def potential_energy(self, world):
        return sum(b.m * self.gmag * b.r[1] for b in world.bodies if not b.fixed)


class Drag(Force):
    """Linear or quadratic drag (Lab 1 Part C). Non-conservative."""
    def __init__(self, k=0.1, quadratic=True):
        self.k = k
        self.quadratic = quadratic
    def apply(self, world):
        for b in world.bodies:
            if b.fixed:
                continue
            speed = np.linalg.norm(b.v)
            if speed == 0:
                continue
            b.add_force(-self.k * speed * b.v if self.quadratic else -self.k * b.v)


class Spring(Force):
    """Hooke's law between two bodies (Lab 3)."""
    def __init__(self, i, j, k=1.0, L0=0.0):
        self.i, self.j, self.k, self.L0 = i, j, k, L0
    def apply(self, world):
        a, b = world.bodies[self.i], world.bodies[self.j]
        d = b.r - a.r
        L = np.linalg.norm(d)
        if L == 0:
            return
        F = self.k * (L - self.L0) * (d / L)
        a.add_force( F)                 # Newton's third law,
        b.add_force(-F)                 # enforced structurally
    def potential_energy(self, world):
        a, b = world.bodies[self.i], world.bodies[self.j]
        L = np.linalg.norm(b.r - a.r)
        return 0.5 * self.k * (L - self.L0)**2


class NewtonGravity(Force):
    """Universal gravitation, all pairs (Lab 5). O(N^2)."""
    def __init__(self, G=1.0, soft=0.0):
        self.G, self.soft = G, soft
    def apply(self, world):
        bs = world.bodies
        for i in range(len(bs)):
            for j in range(i + 1, len(bs)):
                a, b = bs[i], bs[j]
                d  = b.r - a.r
                r2 = d.dot(d) + self.soft**2
                r  = np.sqrt(r2)
                if r == 0:
                    continue
                F = self.G * a.m * b.m / r2 * (d / r)
                a.add_force( F)
                b.add_force(-F)
    def potential_energy(self, world):
        bs, U = world.bodies, 0.0
        for i in range(len(bs)):
            for j in range(i + 1, len(bs)):
                r = np.linalg.norm(bs[j].r - bs[i].r)
                if r > 0:
                    U -= self.G * bs[i].m * bs[j].m / r
        return U


class PivotGravityTorque(Force):
    """Physical pendulum (Lab 6):  tau = -m g d sin(theta)."""
    def __init__(self, idx, m, g, d):
        self.idx, self.m, self.g, self.d = idx, m, g, d
    def apply(self, world):
        b = world.bodies[self.idx]
        b.tau += -self.m * self.g * self.d * np.sin(b.theta)
    def potential_energy(self, world):
        b = world.bodies[self.idx]
        return self.m * self.g * self.d * (1 - np.cos(b.theta))


print("Forces defined:", [c.__name__ for c in
      (Gravity, Drag, Spring, NewtonGravity, PivotGravityTorque)])

### The third law is structural, not optional

Look at `Spring.apply` and `NewtonGravity.apply`: each computes **one** force vector and applies `+F` to one body and `-F` to the other. Newton's third law is not something the engine checks &mdash; it is something the engine *cannot violate*, because there is only one number.

That is why `world.momentum()` is conserved to machine precision in every internal-force simulation below. Any pair force you add later inherits the guarantee for free, provided you write it the same way.

---
## 3 &nbsp; `Integrator` — swap the method, not the simulation

All three share one signature: `step(world, dt)`. That is the whole interface.

Velocity Verlet needs the acceleration from the *previous* step, so it caches `_a` on each body and the world tracks whether it has been primed. That bookkeeping is hidden inside the integrator &mdash; the world and the forces never learn about it.

In [ ]:
class Euler:
    """Explicit Euler. Included so you can watch it fail."""
    name = "Euler"
    def step(self, world, dt):
        world.compute_forces()
        for b in world.bodies:
            if b.fixed:
                continue
            a, alpha = b.f * b.inv_m, b.tau * b.inv_I
            b.r     += b.v * dt          # uses OLD velocity
            b.v     += a * dt
            b.theta += b.omega * dt
            b.omega += alpha * dt


class EulerCromer:
    """Symplectic Euler. One line different -- velocity updates first."""
    name = "Euler-Cromer"
    def step(self, world, dt):
        world.compute_forces()
        for b in world.bodies:
            if b.fixed:
                continue
            b.v     += (b.f * b.inv_m) * dt
            b.r     += b.v * dt          # uses NEW velocity
            b.omega += (b.tau * b.inv_I) * dt
            b.theta += b.omega * dt


class VelocityVerlet:
    """Second-order symplectic. The workhorse for orbits and oscillators."""
    name = "Velocity Verlet"
    def step(self, world, dt):
        bs = world.bodies

        if not world._primed:                       # one-time acceleration prime
            world.compute_forces()
            for b in bs:
                b._a  = b.f * b.inv_m
                b._al = b.tau * b.inv_I
            world._primed = True

        for b in bs:                                # drift with half-step accel
            if b.fixed:
                continue
            b.r     += b.v * dt + 0.5 * b._a  * dt*dt
            b.theta += b.omega * dt + 0.5 * b._al * dt*dt

        world.compute_forces()                      # forces at the NEW positions

        for b in bs:                                # kick with averaged accel
            if b.fixed:
                continue
            a_new, al_new = b.f * b.inv_m, b.tau * b.inv_I
            b.v     += 0.5 * (b._a  + a_new ) * dt
            b.omega += 0.5 * (b._al + al_new) * dt
            b._a, b._al = a_new, al_new


print("Integrators:", [c.name for c in (Euler, EulerCromer, VelocityVerlet)])

---
## 4 &nbsp; Constraints — collisions as a separate stage

Collisions are not forces. A force produces an acceleration integrated over `dt`; a collision delivers an **impulse** that changes velocity discontinuously. Mixing them produces tunnelling and jitter.

So constraints run as their own stage, **after** integration, and expose `resolve(world)`.

Two details that are easy to get wrong, both from Week 4:

- **The separating check `if vn > 0: return`.** Without it, an overlapping pair that is already moving apart gets "resolved" again every step, pumping energy in. Symptom: lowering $e$ *increases* your collision count.
- **Positional correction splits by inverse mass.** Pushing bodies apart by `overlap * inv_m / (inv_m_a + inv_m_b)` moves the light one more, leaves fixed bodies alone, and adds no momentum.

In [ ]:
def resolve_pair(a, b, e=1.0):
    """Impulse + positional correction for two overlapping disks."""
    d    = b.r - a.r
    dist = np.linalg.norm(d)
    if dist == 0 or dist > a.radius + b.radius:
        return False                      # no contact

    n     = d / dist                      # unit line of centres
    v_rel = b.v - a.v
    vn    = v_rel.dot(n)
    if vn > 0:
        return False                      # already separating -- CRITICAL

    im_a, im_b = a.inv_m, b.inv_m
    if im_a + im_b == 0:
        return False                      # two fixed bodies

    J = -(1 + e) * vn / (im_a + im_b)     # scalar impulse
    a.v -= J * im_a * n
    b.v += J * im_b * n

    overlap = a.radius + b.radius - dist  # de-penetrate, split by inverse mass
    a.r -= n * overlap * (im_a / (im_a + im_b))
    b.r += n * overlap * (im_b / (im_a + im_b))
    return True


class CircleCollisions:
    """Narrow phase over all pairs. O(N^2) -- fine to ~200 bodies."""
    def __init__(self, e=1.0):
        self.e = e
        self.count = 0
    def resolve(self, world):
        bs = world.bodies
        for i in range(len(bs)):
            for j in range(i + 1, len(bs)):
                if bs[i].radius > 0 and bs[j].radius > 0:
                    if resolve_pair(bs[i], bs[j], self.e):
                        self.count += 1


class BoxWalls:
    """Axis-aligned container. NOTE: walls are EXTERNAL -- they break momentum."""
    def __init__(self, xmin, xmax, ymin, ymax, e=1.0):
        self.bounds = (xmin, xmax, ymin, ymax)
        self.e = e
        self.count = 0
    def resolve(self, world):
        xmin, xmax, ymin, ymax = self.bounds
        for b in world.bodies:
            if b.fixed:
                continue
            R = b.radius
            if b.r[0] - R < xmin:
                b.r[0] = xmin + R; b.v[0] = -self.e * b.v[0]; self.count += 1
            if b.r[0] + R > xmax:
                b.r[0] = xmax - R; b.v[0] = -self.e * b.v[0]; self.count += 1
            if b.r[1] - R < ymin:
                b.r[1] = ymin + R; b.v[1] = -self.e * b.v[1]; self.count += 1
            if b.r[1] + R > ymax:
                b.r[1] = ymax - R; b.v[1] = -self.e * b.v[1]; self.count += 1


print("Constraints defined: CircleCollisions, BoxWalls")

---
## 5 &nbsp; `World` — the five-stage loop

Everything converges here. One `step(dt)`, five stages, always in this order:

1. **clear** every force accumulator
2. **apply** every force in the registry
3. **integrate** with the chosen method
4. **resolve** constraints
5. **advance** the clock

Stages 1 and 2 live inside `compute_forces()`, because Velocity Verlet needs to call them *twice* per step. Keeping them together is what lets all three integrators share one interface.

In [ ]:
class World:
    """Owns bodies, forces, constraints, the integrator, and the clock."""

    def __init__(self, integrator=None, forces=None, bodies=None, constraints=None):
        self.bodies      = list(bodies or [])
        self.forces      = list(forces or [])
        self.constraints = list(constraints or [])
        self.integrator  = integrator or EulerCromer()
        self.t           = 0.0
        self._primed     = False            # Verlet acceleration cache valid?
        self.recorders   = {}
        self.history     = {}

    # --- scene construction ---
    def add(self, body):
        self.bodies.append(body); self._primed = False; return body

    def add_force(self, force):
        self.forces.append(force); self._primed = False; return force

    def add_constraint(self, c):
        self.constraints.append(c); return c

    # --- stages 1 + 2 ---
    def compute_forces(self):
        for b in self.bodies:
            b.clear()
        for f in self.forces:
            f.apply(self)                   # polymorphism: never asks "which force?"

    # --- the loop ---
    def step(self, dt):
        self.integrator.step(self, dt)      # stages 1-3
        for c in self.constraints:          # stage 4
            c.resolve(self)
        self.t += dt                        # stage 5

    # --- diagnostics ---
    def kinetic_energy(self):
        return sum(b.ke() for b in self.bodies)
    def potential_energy(self):
        return sum(f.potential_energy(self) for f in self.forces)
    def total_energy(self):
        return self.kinetic_energy() + self.potential_energy()
    def momentum(self):
        return sum((b.p() for b in self.bodies), np.zeros(2))
    def angular_momentum(self, origin=(0, 0)):
        return sum(b.L_about(origin) for b in self.bodies)

    # --- recording ---
    def record(self, name, fn):
        """Register a quantity to capture. fn(world) -> number or array."""
        self.recorders[name] = fn
        self.history[name]   = []

    def capture(self):
        for k, fn in self.recorders.items():
            self.history[k].append(fn(self))

    def run(self, dt, t_end, capture_every=1):
        self.capture()
        for i in range(int(round(t_end / dt))):
            self.step(dt)
            if (i + 1) % capture_every == 0:
                self.capture()
        return {k: np.array(v) for k, v in self.history.items()}


print("World defined.")
print("\nEngine complete. Total:", 4, "concepts,", len([Body, World]) + 5 + 3 + 2, "classes.")

### Recording is a registry too

`world.record(name, fn)` stores a *function*, not a value. The engine never hard-codes which quantities matter:

```python
w.record('E',  lambda W: W.total_energy())
w.record('x1', lambda W: W.bodies[1].r[0])
w.record('sep', lambda W: np.linalg.norm(W.bodies[1].r - W.bodies[0].r))
```

Same pattern as the force registry, applied to diagnostics. In Block 3 this becomes your live HUD &mdash; the recorder functions survive the port to JavaScript almost unchanged.

---
# 6 &nbsp; Validation: reproduce Labs 1&ndash;6

An engine you cannot validate is a rewrite you cannot trust. Each section below rebuilds an earlier lab **using only engine calls** &mdash; no hand-written loops &mdash; and checks the numbers against results you already verified.

This is **regression testing**: proving that new code reproduces trusted old results.

## 6.1 &nbsp; Labs 1&ndash;2: projectile

Gravity only. Verlet is *exact* for a constant force, so energy drift should be pure round-off and the range should match $R = v_0^2\sin 2\theta / g$ to the interpolation error.

In [ ]:
v0, angle = 25.0, 45.0
th = np.radians(angle)

w = World(integrator=VelocityVerlet())
w.add(Body(m=1.0, r=(0, 0), v=(v0*np.cos(th), v0*np.sin(th)), name="ball"))
w.add_force(Gravity(9.81))

w.record('t', lambda W: W.t)
w.record('x', lambda W: W.bodies[0].r[0])
w.record('y', lambda W: W.bodies[0].r[1])
w.record('E', lambda W: W.total_energy())

h = w.run(dt=0.001, t_end=3.606)
x, y, E = h['x'], h['y'], h['E']

i    = np.where(y < 0)[0][0]                       # interpolate the landing
frac = y[i-1] / (y[i-1] - y[i])
R    = x[i-1] + frac * (x[i] - x[i-1])

print("PROJECTILE  v0 = 25 m/s at 45 deg, Velocity Verlet, dt = 1e-3")
print(f"  E0                = {E[0]:.10f} J")
print(f"  max|dE|/E0        = {np.max(np.abs(E-E[0]))/abs(E[0]):.3e}")
print(f"  range (simulated) = {R:.4f} m")
print(f"  range (theory)    = {v0**2*np.sin(2*th)/9.81:.4f} m")

fig, ax = plt.subplots(1, 2)
ax[0].plot(x[:i+1], y[:i+1], color=UT_ORANGE, lw=2)
ax[0].axhline(0, color='k', lw=0.8); ax[0].set_xlabel('x (m)')
ax[0].set_ylabel('y (m)'); ax[0].set_title('Trajectory')
ax[1].plot(h['t'][:i+1], (E[:i+1]-E[0])/E[0], color='#17a2b8')
ax[1].set_xlabel('t (s)'); ax[1].set_ylabel(r'$\Delta E/E_0$')
ax[1].set_title('Energy drift (round-off only)')
plt.tight_layout(); plt.show()

## 6.2 &nbsp; Lab 3: the integrator showdown

Same scene, three integrators, **one line changed**. The spring makes the force position-dependent, which switches on the feedback loop that uniform gravity hides (Week 2, §5.5).

In [ ]:
T_sho = 2*np.pi            # k = m = 1
rows = []

for Int in (Euler, EulerCromer, VelocityVerlet):
    w = World(integrator=Int())                        # <-- the only change
    w.add(Body(m=1.0, r=(0, 0), fixed=True))
    w.add(Body(m=1.0, r=(1, 0), v=(0, 0)))
    w.add_force(Spring(0, 1, k=1.0, L0=0.0))
    w.record('t', lambda W: W.t)
    w.record('x', lambda W: W.bodies[1].r[0])
    w.record('E', lambda W: W.total_energy())
    h = w.run(dt=0.05, t_end=10*T_sho)
    rows.append((Int.name, h))

print("SHO  k = m = 1, x0 = 1, 10 periods, dt = 0.05")
print(f"{'integrator':<18}{'max|dE|/E0':>14}{'final x':>12}")
for name, h in rows:
    E = h['E']
    print(f"{name:<18}{np.max(np.abs(E-E[0]))/abs(E[0]):>14.4e}{h['x'][-1]:>12.4f}")

fig, ax = plt.subplots(1, 2)
for (name, h), c in zip(rows, ['#dc3545', UT_ORANGE, '#28a745']):
    ax[0].plot(h['t'], h['x'], label=name, lw=1.2, color=c)
    ax[1].semilogy(h['t'], np.abs(h['E']-h['E'][0])/h['E'][0] + 1e-18,
                   label=name, lw=1.2, color=c)
ax[0].set_xlabel('t (s)'); ax[0].set_ylabel('x (m)'); ax[0].set_title('Displacement')
ax[0].set_ylim(-3, 3); ax[0].legend(fontsize=8)
ax[1].set_xlabel('t (s)'); ax[1].set_ylabel(r'$|\Delta E|/E_0$')
ax[1].set_title('Energy error'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

Euler's error reaches **22&times; the total energy** &mdash; the amplitude more than doubles over ten periods. Euler-Cromer stays bounded at 2.6%, Verlet at 0.06%.

Changing `World(integrator=Euler())` to `World(integrator=VelocityVerlet())` is a 4,000&times; accuracy improvement and a one-token edit. That is what the architecture buys.

## 6.3 &nbsp; Lab 5: orbits and Kepler's third law

A central force conserves angular momentum **exactly** for a symplectic integrator &mdash; watch the $\Delta L$ column.

In [ ]:
print("ORBIT  G = M = 1, r0 = (1,0), v0 = (0,0.8), dt = 1e-3, t = 20")
print(f"{'integrator':<18}{'max|dE/E0|':>13}{'max|dL/L0|':>13}{'r_min':>9}{'r_max':>9}")

trajs = []
for Int in (Euler, EulerCromer, VelocityVerlet):
    w = World(integrator=Int())
    w.add(Body(m=1.0,  r=(0, 0), fixed=True, name="star"))
    w.add(Body(m=1e-6, r=(1, 0), v=(0, 0.8), name="planet"))
    w.add_force(NewtonGravity(G=1.0))
    w.record('x', lambda W: W.bodies[1].r[0])
    w.record('y', lambda W: W.bodies[1].r[1])
    w.record('r', lambda W: np.linalg.norm(W.bodies[1].r))
    w.record('E', lambda W: W.total_energy())
    w.record('L', lambda W: W.angular_momentum())
    h = w.run(dt=0.001, t_end=20.0)
    E, L, r = h['E'], h['L'], h['r']
    print(f"{Int.name:<18}{np.max(np.abs(E-E[0]))/abs(E[0]):>13.3e}"
          f"{np.max(np.abs(L-L[0]))/abs(L[0]):>13.3e}{r.min():>9.4f}{r.max():>9.4f}")
    trajs.append((Int.name, h))

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, (name, h), c in zip(ax, trajs, ['#dc3545', UT_ORANGE, '#28a745']):
    a.plot(h['x'], h['y'], lw=0.6, color=c)
    a.plot(0, 0, '*', ms=14, color='gold', mec='k')
    a.set_aspect('equal'); a.set_title(name); a.set_xlabel('x')
ax[0].set_ylabel('y')
plt.tight_layout(); plt.show()

In [ ]:
# Kepler's third law, measured
from scipy.signal import argrelmin

w = World(integrator=VelocityVerlet())
w.add(Body(m=1.0,  r=(0, 0), fixed=True))
w.add(Body(m=1e-6, r=(1, 0), v=(0, 0.8)))
w.add_force(NewtonGravity(G=1.0))
w.record('t', lambda W: W.t)
w.record('r', lambda W: np.linalg.norm(W.bodies[1].r))
h = w.run(dt=0.0005, t_end=20.0)
r, t = h['r'], h['t']

a_meas = (r.min() + r.max()) / 2
E_sp   = 0.5*0.8**2 - 1.0            # specific orbital energy
a_th   = -1 / (2*E_sp)
idx    = argrelmin(r, order=50)[0]
T_meas = np.mean(np.diff(t[idx]))

print("KEPLER CHECK  (Velocity Verlet, dt = 5e-4)")
print(f"  perihelion            = {r.min():.6f}")
print(f"  aphelion              = {r.max():.6f}")
print(f"  a  (measured)         = {a_meas:.6f}")
print(f"  a  (theory, -1/2E)    = {a_th:.6f}")
print(f"  T  (measured)         = {T_meas:.6f}")
print(f"  T  (Kepler 2*pi*a^3/2)= {2*np.pi*a_th**1.5:.6f}")

Angular momentum is conserved to $\sim 10^{-15}$ for both symplectic methods &mdash; **machine zero**. That is not luck: a central force exerts no torque about the centre, and symplectic integrators preserve that structure exactly. Euler loses 5.5% of $L$, which is why its orbit visibly precesses.

Semi-major axis and period match theory to six figures.

## 6.4 &nbsp; Lab 4: collisions

Two runs, one scene. **Walls off** gives an isolated system where momentum must be conserved to machine precision. **Walls on** adds an external force &mdash; momentum is no longer conserved, but at $e = 1$ energy still is, because an elastic wall reverses a velocity component and does no work.

In [ ]:
def collision_scene(e, walls=True):
    coll = CircleCollisions(e=e)
    cons = [coll]
    wl   = BoxWalls(0, 10, 0, 10, e=1.0) if walls else None
    if wl: cons.append(wl)
    w = World(integrator=EulerCromer(), constraints=cons)
    for p, v in zip([(2,2), (8,2), (2,8), (8,8)],
                    [(3,2), (-2,3), (2.5,-1.5), (-3,-2)]):
        w.add(Body(m=1.0, r=p, v=v, radius=0.5))
    return w, coll, wl

print("ISOLATED (walls off), t = 6 s -- momentum must hold")
print(f"{'e':>5}{'|dp|/|p0|':>14}{'KE retained':>14}{'pair hits':>11}")
for e in (1.0, 0.9, 0.5, 0.0):
    w, coll, _ = collision_scene(e, walls=False)
    p0, KE0 = w.momentum().copy(), w.kinetic_energy()
    w.run(dt=0.002, t_end=6.0)
    dp = np.linalg.norm(w.momentum()-p0) / np.linalg.norm(p0)
    print(f"{e:>5.1f}{dp:>14.3e}{100*w.kinetic_energy()/KE0:>13.2f}%{coll.count:>11d}")

print("\nIN A BOX (walls on), t = 20 s -- walls are EXTERNAL")
print(f"{'e':>5}{'KE retained':>14}{'pair hits':>11}{'wall hits':>11}")
for e in (1.0, 0.9, 0.5, 0.0):
    w, coll, wl = collision_scene(e)
    KE0 = w.kinetic_energy()
    w.run(dt=0.002, t_end=20.0)
    print(f"{e:>5.1f}{100*w.kinetic_energy()/KE0:>13.3f}%{coll.count:>11d}{wl.count:>11d}")

Two things to read carefully.

**Momentum is conserved to $10^{-16}$ at every $e$** &mdash; even at $e = 0$, where 44% of the kinetic energy is gone. That is the Week 4 asymmetry made quantitative: momentum has nowhere to hide, energy does.

**The box column is not monotonic in $e$.** At $e = 0$ the balls retain *more* energy than at $e = 0.5$. Not a bug: perfectly inelastic bodies clump into slow-moving aggregates that stop meeting, so the dissipation mechanism shuts itself off. A student reporting a smooth trend here is reporting what they expected rather than what they measured.

## 6.5 &nbsp; Lab 6: the physical pendulum

Rotational state, driven by a torque instead of a force. The engine does not notice the difference &mdash; the same `step()` handles it, because `theta`/`omega`/`tau`/`inv_I` are exact mirrors of `r`/`v`/`f`/`inv_m`.

Validated against the **exact** period $T = \dfrac{4}{\Omega}K\!\left(\sin^2\frac{\theta_0}{2}\right)$, not the small-angle formula.

In [ ]:
m_rod, L_rod, g = 1.0, 1.0, 9.81
I_rod = m_rod * L_rod**2 / 3
d_cm  = L_rod / 2
Omega = np.sqrt(m_rod * g * d_cm / I_rod)

print(f"Rod about one end:  Omega = sqrt(3g/2L) = {Omega:.6f} rad/s")
print(f"                    T0    = {2*np.pi/Omega:.6f} s\n")
print(f"{'theta0':>8}{'T (engine)':>13}{'T (exact)':>13}{'rel err':>11}{'T/T0':>8}")

for th0 in (0.1, 0.3, 1.0, 2.0, 3.0):
    w = World(integrator=EulerCromer())
    w.add(Body(m=m_rod, r=(0, 0), I=I_rod, theta=th0, omega=0.0, name="rod"))
    w.add_force(PivotGravityTorque(0, m_rod, g, d_cm))
    w.record('t',  lambda W: W.t)
    w.record('th', lambda W: W.bodies[0].theta)
    h = w.run(dt=1e-4, t_end=12.0)

    th, t = h['th'], h['t']
    s     = np.sign(th)
    cross = np.where((s[:-1] > 0) & (s[1:] <= 0))[0]
    tc    = [t[i] + th[i]/(th[i]-th[i+1])*(t[i+1]-t[i]) for i in cross]
    T_sim = np.mean(np.diff(tc))
    T_ex  = 4/Omega * ellipk(np.sin(th0/2)**2)

    print(f"{th0:>8.1f}{T_sim:>13.6f}{T_ex:>13.6f}"
          f"{abs(T_sim-T_ex)/T_ex:>11.2e}{T_sim/(2*np.pi/Omega):>8.4f}")

Nine-decimal agreement with the exact elliptic solution at every amplitude, including $\theta_0 = 3.0$ rad where the period is **2.57&times;** the small-angle value. The engine reproduces Lab 6 without a single line of rotation-specific loop code.

---
# 7 &nbsp; The payoff: a scene you could not have written in Lab 1

The figure-eight three-body orbit (Chenciner & Montgomery, 2000): three equal masses chasing each other around a single closed curve. It exists only for one precise set of initial conditions, and it is a brutal test &mdash; any energy or momentum error destroys it within a few periods.

Eleven lines of scene setup. No new engine code.

In [ ]:
x1, y1   = 0.97000436, -0.24308753
vx3, vy3 = -0.93240737, -0.86473146
T_fig    = 6.32591398

w = World(integrator=VelocityVerlet())
w.add(Body(m=1, r=( x1,  y1), v=(-vx3/2, -vy3/2), name="A"))
w.add(Body(m=1, r=(-x1, -y1), v=(-vx3/2, -vy3/2), name="B"))
w.add(Body(m=1, r=(  0,   0), v=( vx3,    vy3  ), name="C"))
w.add_force(NewtonGravity(G=1.0))

for i in range(3):
    w.record(f'x{i}', lambda W, i=i: W.bodies[i].r[0])
    w.record(f'y{i}', lambda W, i=i: W.bodies[i].r[1])
w.record('t', lambda W: W.t)
w.record('E', lambda W: W.total_energy())
w.record('p', lambda W: np.linalg.norm(W.momentum()))
w.record('L', lambda W: W.angular_momentum())

h = w.run(dt=1e-4, t_end=4*T_fig, capture_every=10)
E, p, L = h['E'], h['p'], h['L']

print("FIGURE-EIGHT THREE-BODY ORBIT   4 periods, Verlet, dt = 1e-4")
print(f"  E0                   = {E[0]:.9f}")
print(f"  max|dE/E0|           = {np.max(np.abs(E-E[0]))/abs(E[0]):.3e}")
print(f"  max|p|  (exact 0)    = {p.max():.3e}")
print(f"  max|dL| (exact 0)    = {np.max(np.abs(L-L[0])):.3e}")
print(f"  |r_A(4T) - r_A(0)|   = {np.linalg.norm(w.bodies[0].r-np.array([x1,y1])):.3e}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for i, c in enumerate([UT_ORANGE, '#17a2b8', '#28a745']):
    ax[0].plot(h[f'x{i}'], h[f'y{i}'], lw=1.0, color=c, alpha=0.8)
    ax[0].plot(w.bodies[i].r[0], w.bodies[i].r[1], 'o', ms=9, color=c, mec='k')
ax[0].set_aspect('equal'); ax[0].set_title('Figure-eight, 4 periods')
ax[0].set_xlabel('x'); ax[0].set_ylabel('y')
ax[1].plot(h['t'], (E-E[0])/abs(E[0]), color=UT_ORANGE, lw=1.2)
ax[1].set_xlabel('t'); ax[1].set_ylabel(r'$\Delta E/E_0$')
ax[1].set_title('Energy drift')
plt.tight_layout(); plt.show()

Total momentum is $4\times10^{-14}$ &mdash; machine zero, guaranteed by the $\pm F$ structure of `NewtonGravity`, not by anything the world checks. Energy drifts by 6 parts in $10^9$ over four periods, and body A returns to within $3\times10^{-5}$ of where it started.

**This is the week's argument in one cell.** Figure-eight initial conditions are not harder to *write* than a projectile &mdash; they were just impossible to *run* before you had an engine.

---
## 8 &nbsp; What it costs: $O(N^2)$

`NewtonGravity` and `CircleCollisions` both loop over all pairs: $N(N-1)/2$ of them. Doubling $N$ should roughly **quadruple** the cost.

In [ ]:
import time

print("NewtonGravity + Verlet, 200 steps")
print(f"{'N':>5}{'pairs':>8}{'time (s)':>11}{'ratio':>8}")
prev = None
for N in (10, 20, 40, 80):
    w   = World(integrator=VelocityVerlet())
    rng = np.random.default_rng(3)
    for _ in range(N):
        w.add(Body(m=1.0, r=rng.uniform(-5, 5, 2), v=rng.uniform(-.1, .1, 2)))
    w.add_force(NewtonGravity(G=1.0, soft=0.5))
    t0 = time.perf_counter()
    for _ in range(200):
        w.step(0.001)
    el = time.perf_counter() - t0
    r  = f"{el/prev:.2f}" if prev else "--"
    print(f"{N:>5}{N*(N-1)//2:>8}{el:>11.4f}{r:>8}")
    prev = el

Ratios cluster near 4, confirming $O(N^2)$.

Two consequences for Block 3, where this runs 60 times a second in a browser:

- **A few hundred bodies is the honest ceiling** for all-pairs. Beyond that you need a spatial grid (collisions) or Barnes&ndash;Hut (gravity) &mdash; both listed as "one new thing" options in the Week 11 project menu.
- **Profile before optimising.** In most scenes the pair loop dominates, but not always. Measure first.

---
## 9 &nbsp; Fixed timestep: the accumulator

In Python you choose `dt` and the loop obeys. In a browser the loop runs at whatever rate the display manages &mdash; 60 fps, 144 fps, or 8 fps when a background tab wakes up.

**Never pass the frame time to `step()`.** Accumulate it and spend it in fixed-size pieces:

```python
accumulator += min(frame_time, 0.25)      # clamp: never spiral
while accumulator >= DT:
    world.step(DT)
    accumulator -= DT
```

Below, the same 10 s of spring motion under three wildly different frame patterns.

In [ ]:
DT, T_TOTAL = 1/240, 10.0

def spring_scene():
    w = World(integrator=VelocityVerlet())
    w.add(Body(m=1.0, r=(0, 0), fixed=True))
    w.add(Body(m=1.0, r=(1.0, 0.0), v=(0.0, 0.0)))
    w.add_force(Spring(0, 1, k=4.0, L0=0.0))
    return w

def patterns(total):
    rng = np.random.default_rng(11)
    out = {"steady 60 fps": np.full(int(total*60), 1/60)}
    j = rng.uniform(1/144, 1/24, 4000); j = j[np.cumsum(j) <= total]
    out["jittery 24-144"] = np.append(j, total - j.sum())
    s = np.full(int(total*60), 1/60); s[::37] = 0.25
    s = s[np.cumsum(s) <= total]
    out["60 fps + stalls"] = np.append(s, total - s.sum())
    return out

exact = np.cos(2.0 * T_TOTAL)        # omega = sqrt(k/m) = 2

print("FIXED TIMESTEP ACCUMULATOR")
print(f"{'frame pattern':<18}{'frames':>8}{'steps':>8}{'x(10 s)':>16}")
for label, frames in patterns(T_TOTAL).items():
    w, acc, steps = spring_scene(), 0.0, 0
    for ft in frames:
        acc += min(ft, 0.25)
        while acc >= DT:
            w.step(DT); acc -= DT; steps += 1
    print(f"{label:<18}{len(frames):>8}{steps:>8}{w.bodies[1].r[0]:>16.10f}")

print("\nNAIVE VARIABLE TIMESTEP  (step once per frame, dt = frame time)")
print(f"{'frame pattern':<18}{'frames':>8}{'x(10 s)':>16}{'error':>12}")
for label, frames in patterns(T_TOTAL).items():
    w = spring_scene()
    for ft in frames:
        w.step(min(ft, 0.25))
    x = w.bodies[1].r[0]
    print(f"{label:<18}{len(frames):>8}{x:>16.10f}{abs(x-exact):>12.2e}")
print(f"\n  exact x(10 s) = {exact:.10f}")

The first two accumulator rows are **identical to ten decimal places** despite one using 600 frames and the other 426. Step count is driven by elapsed time, not frame count.

The third row differs slightly &mdash; it ran 2399 steps instead of 2400, leaving a partial `DT` in the accumulator, so it is reporting the state at $t = 9.9958$ s rather than $10$ s. That leftover is real and expected; in Block 3 you render it away by interpolating with $\alpha = \texttt{acc}/DT$.

The naive version degrades with frame quality: 60 fps is off by $8\times10^{-4}$, stalls by $5\times10^{-2}$ &mdash; **60&times; worse**, from the same physics.

> **Why a spring and not a projectile?** Try this with uniform gravity and variable $dt$ looks fine, because Verlet is exact for a constant force. Same degeneracy as the Week 2 §5.5 insert: a position-independent force cannot expose an integrator flaw. Choosing a revealing test case is part of the test.

---
## 10 &nbsp; The architecture test

Before you call your Lab 7 engine done, run these four checks.

**1. Could you add a new force without editing `World`?** Write one now &mdash; a constant wind, say &mdash; and confirm it drops in.

In [ ]:
class Wind(Force):
    """Constant force on every non-fixed body. Non-conservative."""
    def __init__(self, F=(1.0, 0.0)):
        self.F = np.array(F, dtype=float)
    def apply(self, world):
        for b in world.bodies:
            if not b.fixed:
                b.add_force(self.F)

w = World(integrator=VelocityVerlet())
w.add(Body(m=1.0, r=(0, 0), v=(10, 20)))
w.add_force(Gravity(9.81))
w.add_force(Wind((-2.0, 0.0)))        # <-- new physics, zero engine edits
w.record('x', lambda W: W.bodies[0].r[0])
w.record('y', lambda W: W.bodies[0].r[1])
h = w.run(dt=0.001, t_end=4.0)

w2 = World(integrator=VelocityVerlet())
w2.add(Body(m=1.0, r=(0, 0), v=(10, 20)))
w2.add_force(Gravity(9.81))
w2.record('x', lambda W: W.bodies[0].r[0])
w2.record('y', lambda W: W.bodies[0].r[1])
h2 = w2.run(dt=0.001, t_end=4.0)

print("Lines added to Body / Force / Integrator / World:  0")
print(f"x at t = 4 s   no wind: {h2['x'][-1]:8.4f} m")
print(f"               2 N headwind: {h['x'][-1]:8.4f} m")

plt.figure(figsize=(7, 4))
plt.plot(h2['x'], h2['y'], lw=2, color=UT_ORANGE, label='gravity only')
plt.plot(h['x'],  h['y'],  lw=2, color='#17a2b8', label='+ 2 N headwind')
plt.axhline(0, color='k', lw=0.8); plt.xlabel('x (m)'); plt.ylabel('y (m)')
plt.legend(); plt.title('New force, no engine changes'); plt.ylim(bottom=-1)
plt.tight_layout(); plt.show()

**2. Could you swap integrators with one token?** Every showdown above did exactly that.

**3. Could you build a brand-new scene without touching the engine?** Sections 6.1&ndash;6.5 and 7 each build a different physical system from the same six classes.

**4. Would this port cleanly to JavaScript?** This is the one students underrate. Check your engine for:

| Python feature | Portability |
|---|---|
| `np.array` 2-vectors | Becomes a tiny `Vec2` class &mdash; fine |
| `@property` for `inv_m` | ES6 `get inv_m()` &mdash; fine |
| Duck-typed `apply(world)` | Identical in JS &mdash; fine |
| `**kwargs`, decorators, metaclasses | **No clean equivalent. Avoid.** |
| Vectorised whole-array NumPy ops | Must be rewritten as loops |
| `lambda W: ...` recorders | Arrow functions &mdash; fine |

Nothing in this notebook uses a Python feature that lacks a direct JavaScript counterpart. That was a design constraint, not an accident.

---

## 11 &nbsp; What ports, and what comes next

| This notebook | Block 3 equivalent |
|---|---|
| `Body` | `engine/body.js` |
| `Gravity`, `Drag`, `Spring`, `NewtonGravity` | `engine/forces.js` |
| `EulerCromer`, `VelocityVerlet` | `engine/integrators.js` |
| `resolve_pair`, `CircleCollisions` | `engine/collisions.js` |
| `World.step()` | `engine/world.js` |
| `world.record(...)` | `render/hud.js` &mdash; live diagnostics |
| `world.run(dt, t_end)` | `requestAnimationFrame` + accumulator |
| *(plots)* | `render/renderer.js` &mdash; the only genuinely new file |

Nine weeks of physics collapse into one architecture, and the browser port adds exactly one new concern: drawing.

> **Design note for Lab 7.** `World` never asks what kind of force, integrator, or constraint it is holding. Every component answers one question and exposes one method. That restraint is what makes the engine extensible &mdash; and it is what Lab 7 is graded on, more than correctness.

---

### Deliverables this week

- **Lab 7** &mdash; build your own engine, reproduce at least three earlier labs, add an original multi-body demo
- **Block 2 Checkpoint** &mdash; the engine is your Stage 1 &rarr; Stage 2 handoff artifact
- **Discussion: Design Decisions** &mdash; post your class diagram; critique two others

Keep it simple. You are going to type this engine again, in JavaScript, in two weeks.